# Multi-turn TODO Chatbot Graph Visualization

`agents/todo_creation/multi_turn/graph.py` 의 컴파일된 LangGraph 를 그대로 그려본다.
코드와 다이어그램이 항상 동기화된다 (수동 mermaid 파일과 달리).

그래프 구조 요약:

- `validate → phase_router`
- `phase_router (gathering) → planner_judge → {plan_generator → tagger | follow_up}`
- `phase_router (reviewing) → edit_agent → {plan_generator → tagger | commit_invoke}`
- `* → present → END`

LLM 노드(planner_judge, follow_up, plan_generator, tagger, edit_agent) 에 `RetryPolicy` 적용 (4개는 `max_attempts=2`, edit_agent 는 `max_attempts=1`).


In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from agents.todo_creation.multi_turn.graph import build_multi_turn_graph

graph = build_multi_turn_graph()
print(graph)

## 1. Mermaid 소스 출력

외부 의존성 없이 항상 동작. 결과 문자열을 https://mermaid.live 에 붙여넣으면 렌더링됨.


In [ ]:
mermaid_src = graph.get_graph().draw_mermaid()
print(mermaid_src)

## 2. Mermaid PNG 인라인 렌더링

mermaid.ink 공개 API 를 호출해 PNG 를 받아 노트북에 표시.
**인터넷 연결 필요.** 차단된 환경이면 셀 3 으로 넘어갈 것.


In [ ]:
from IPython.display import Image, display

try:
    png_bytes = graph.get_graph().draw_mermaid_png()
    display(Image(png_bytes))
except Exception as exc:
    print(f"PNG 렌더링 실패: {exc!r}")
    print("→ 인터넷 차단 또는 mermaid.ink 응답 오류. 셀 1 의 텍스트 출력을 사용하세요.")

## 3. 노드/엣지 원본 데이터 확인

어떤 노드/엣지가 등록되었는지 직접 확인 — `graph.py` 수정 후 검증용. 조건부 엣지는 `[conditional]` 마커로 표시된다.


In [ ]:
g = graph.get_graph()

print("=== Nodes ===")
for node_id in g.nodes:
    print(f"  - {node_id}")

print("\n=== Edges ===")
for edge in g.edges:
    cond = " [conditional]" if edge.conditional else ""
    label = f" ({edge.data})" if edge.data else ""
    print(f"  - {edge.source} → {edge.target}{cond}{label}")